# Destination-choice + stay/move model via torch-choice

torch-choice port of `us/modeling_mnl.ipynb` (the Biogeme model). Same choice structure as the
Biogeme notebook: `alt=0` is staying, `alt=1..50` are the move alternatives (one row per person
per alternative in the reshaped data), and every person's utility is `V[0]` (stay) or `V[i]`
(move to alternative `i`), matching the Biogeme notebook's `V` dictionary.

**Coefficient sharing is preserved.** A number of Biogeme `Beta`s (e.g. `c_proportion_same_age_18_34`)
are used in *both* `V[0]` and `V[i]` with different variable expressions on each side (comparing the
person to their origin vs. to the candidate destination) but the *same* estimated coefficient. torch-choice
has no direct equivalent of Biogeme's named, reusable `Beta` objects, so this is reproduced by giving the
stay-context and move-context expressions for each shared concept the same column name in the reshaped
long-format table below (e.g. `proportion_same_age_18_34`) — since torch-choice's `ConditionalLogitModel`
assigns exactly one coefficient per input column, this ties the two contexts to one shared parameter,
exactly like the shared `Beta` does in Biogeme.

**Deviations from `modeling_mnl.ipynb` as currently written**, found while porting (the source notebook
does not currently run against the current `us_estdata_2018.parquet` schema either — these look like a
still-in-progress rename, not settled model changes):
- Cell 5's `ALT_VARYING_SUFFIXES` there still lists `OWN_RACE_PCT` / `OWN_GROUP_PCT`, which no longer exist
  in the parquet schema (they were renamed `OWN_RACE_ETH_PCT` / `OWN_NAICS_GROUP_PCT` — the names its own
  `V[i]` utility formulas in cell 23 already use). This port reads the new names directly.
- Cell 23 has `Variable(f"{alt}OWN_GROUPOWN_NAICS_GROUP_PCT_PCT")` for `c_proportion_same_naics_goods_trade`
  — a mangled column name (looks like a partially-applied find/replace). Treated here as
  `f"{alt}OWN_NAICS_GROUP_PCT"`, matching the other four `same_naics_*` terms in the same block.
- `c_proportion_same_race_white` and `c_proportion_same_race_other` reference columns that are not in
  `INDIV_COLS` at all (`WHITE`, `"Proportion of people White.ORIG"`), and the `V[i]` copy of the white-race
  term isn't even alternative-varying (no `{alt}` prefix — identical expression to `V[0]`'s). The `V[0]`
  version of the "other race" term also references a literally truncated column name (`"Proportion of "`).
  These three terms look like leftovers from the in-progress race/ethnicity recategorization (see the
  `add more comprehensive categorization of race/ethnicity, rename` commit) and are **omitted** here rather
  than guessed at — see the note near the shared-terms cell below.
- `ALT{i}_STATE` (and `CHOSEN` / `ALT{i}_PUMA`) are `string`-dtype in the current parquet, not
  `category`/`object`, so Biogeme's `df_train.select_dtypes(["number"])` (cell 9) would silently drop
  `STATE` entirely, and `Variable(f"{alt}STATE")` in cell 23 would then fail at estimation time. Since this
  port builds its own tensors instead of a Biogeme `Database`, `STATE` is just cast to `int` directly.
- The stay-side log-population offset (`log(Total Population....ORIG)`, the origin's population) is
  included alongside the move-side one (`log(ALT{i}_TOT_POP)`, the destination's population) — both are
  additive terms with an implicit coefficient of 1 (not estimated), matching Biogeme's un-parameterized
  `log(...)` calls in `V[0]` and `V[i]`.

**Optimizer note:** the `torch-choice` version pinned in this project (`1.0.6`) only exposes
`{SGD, Adagrad, Adadelta, Adam}` via `torch_choice.utils.run_helper.run` — no BFGS/LBFGS like Biogeme
uses, and no `.fit()`/device-aware API (that landed in `1.0.7`, which currently can't be installed here:
it pins `numpy<2.0`, conflicting with this project's `numpy 2.5.1`). Adam is used below; full-batch,
first-order optimization on a model this size will need more epochs and more tuning to converge than
Biogeme's BFGS does.


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch_choice.data import ChoiceDataset
from torch_choice.model import ConditionalLogitModel
from torch_choice.utils.run_helper import run as tc_run


In [ ]:
year = 2018
num_alternatives = 50


### Read data, only keep used columns

In [ ]:
# same columns= restriction technique as modeling_mnl.ipynb: read only the columns V[0]/V[i]
# actually reference, instead of the full ~3,800-column file, since most of it is unused census
# fields. ALT_VARYING_SUFFIXES covers the ALT{i}_<suffix> destination columns V[i] uses;
# INDIV_COLS covers everything else, plus CHOSEN/STAY/ALT{i}_PUMA needed to build ALT_CHOICE below.
# NOTE: OWN_RACE_ETH_PCT / OWN_NAICS_GROUP_PCT here, not OWN_RACE_PCT / OWN_GROUP_PCT as modeling_mnl.ipynb's
# cell 5 currently has -- see the intro markdown cell for why.
ALT_VARYING_SUFFIXES = [
    "ALT_COMMUTE_PCT",
    "CBSA",
    "COLLEGE_PCT",
    "DIST",
    "ENT_JOBS_PCT",
    "FOREIGN_BORN_PCT",
    "HH_MED_INC",
    "HH_WITH_CHILD_PCT",
    "HOUSE_VACANCY_PCT",
    "MED_HOUSE_VALUE_OVER_MED_HH_INC",
    "MED_RENT_PCT_HH_INC",
    "MED_TRAVEL_TIME",
    "MIL_PCT",
    "OWN_AGE_PCT",
    "OWN_NAICS_GROUP_PCT",
    "OWN_RACE_ETH_PCT",
    "STATE",
    "TOT_POP",
    "TYPE",
    "UNEMP_RATE",
]

INDIV_COLS = [
    "CHOSEN",
    "STAY",
    "AAPI",
    "AGE_18_22",
    "AGE_18_34",
    "AGE_23_29",
    "AGE_30_39",
    "AGE_35_64",
    "AGE_40_49",
    "AGE_50_64",
    "AGE_OVER_65",
    "BLACK",
    "CHILD",
    "CHILD_6_TO_17",
    "CHILD_UNDER_6",
    "EDU_BACHELORS",
    "EDU_HIGH",
    "EDU_NOHIGH",
    "FOREIGN",
    "House vacancy proportion.ORIG",
    "INDIAN",
    "IN_COLLEGE",
    "IN_MILITARY",
    "LATINO",
    "MARRIED_MORE_THAN_YEAR",
    "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG",
    "Median gross rent as a percentage of household income.ORIG",
    "Median house value over median household income.ORIG",
    "Median travel time.ORIG",
    "NAICS_AGR_EXT",
    "NAICS_GOODS_TRADE",
    "NAICS_GOVT",
    "NAICS_GROUP_PCT_AGR_EXT.ORIG",
    "NAICS_GROUP_PCT_GOODS_TRADE.ORIG",
    "NAICS_GROUP_PCT_GOVT.ORIG",
    "NAICS_GROUP_PCT_HIGH_ED.ORIG",
    "NAICS_GROUP_PCT_LICENSE.ORIG",
    "NAICS_HIGH_ED",
    "NAICS_LICENSE",
    "NAME_NUM.ORIG",
    "POBP",
    "Proportion alternative commute.ORIG",
    "Proportion foreign born.ORIG",
    "Proportion of households with children.ORIG",
    "Proportion of people 18-34.ORIG",
    "Proportion of people 35-64.ORIG",
    "Proportion of people 65+.ORIG",
    "Proportion of people AAPI.ORIG",
    "Proportion of people Black.ORIG",
    "Proportion of people Indian.ORIG",
    "Proportion of people Latino.ORIG",
    "Proportion of people in college.ORIG",
    "Proportion of people in military.ORIG",
    "RECENTLY_MARRIED",
    "RECENTLY_WIDOWED_OR_DIVORCED",
    "SINGLE_PARENT",
    "ST",
    "TYPE_NUM.ORIG",
    "Unemployment rate.ORIG",
    "WORK1_MAR",
    "WORK2_MAR",
    "Total Population.Total Population.SE_A00001_001.ORIG",
    "Proportion of entertainment jobs.ORIG",
]

needed_cols = (
    INDIV_COLS
    + [f"ALT{i}_PUMA" for i in range(1, num_alternatives + 1)]
    + [
        f"ALT{i}_{suf}"
        for i in range(1, num_alternatives + 1)
        for suf in ALT_VARYING_SUFFIXES
    ]
)
needed_cols = list(dict.fromkeys(needed_cols))
print(f"reading {len(needed_cols)} columns")

df = pd.read_parquet(f"../data/us_estdata_{year}.parquet", columns=needed_cols)
print(df.shape)


### Clean data, make everything numeric

In [ ]:
cat_cols = df.dtypes[df.dtypes == "category"].keys()
df[cat_cols] = df[cat_cols].apply(lambda x: x.astype(int))

obj_cols = list(df.dtypes[df.dtypes == "object"].keys())
df[obj_cols] = df[obj_cols].apply(lambda x: x.astype(int))

# ALT{i}_STATE is pandas "string" dtype in the current parquet (not category/object, so the two
# conversions above miss it) -- cast explicitly so it's comparable to the int64 ST/POBP columns
# in the same_state / birthstate terms below. CHOSEN and ALT{i}_PUMA are also "string" dtype, but
# stay that way on purpose: they're only ever compared to each other, both as strings.
state_cols = [f"ALT{i}_STATE" for i in range(1, num_alternatives + 1)]
df[state_cols] = df[state_cols].astype(int)

df["person_id"] = np.arange(len(df))


### Creating alternatives

In [ ]:
# defining the chosen alternative for each person explicitly, same convention as modeling_mnl.ipynb:
# each person has chosen from alternatives 0-num_alternatives, 0 represents staying and 1-num_alternatives
# represent moving to the PUMAs each represents. Movers match their true chosen destination by PUMA;
# stayers are always alt=0.
df["ALT_CHOICE"] = 0
for i in range(1, num_alternatives + 1):
    var = f"ALT{i}_PUMA"
    df["ALT_CHOICE"] = np.where(df[var] == df["CHOSEN"], i, df["ALT_CHOICE"])
df["ALT_CHOICE"] = np.where(df["STAY"] == 1, 0, df["ALT_CHOICE"])
assert (df["ALT_CHOICE"] > 0).sum() == (df["STAY"] == 0).sum(), (
    "every mover must match exactly one ALT{i}_PUMA"
)


### Reshape to long format

torch-choice (like xlogit) wants one row per `(person, alternative)` pair rather than Biogeme's
wide format with a `V[i]` formula evaluated directly against the wide dataframe. `wide_to_long` below
is a small local replacement for `xlogit.utils.wide_to_long` (xlogit is no longer a project dependency)
that does the same `ALT{i}_<suffix>` -> `<suffix>` melt.

Move and stay don't share a formula shape (different variable *expressions* entirely on each side, even
for the shared-coefficient terms), so the move block is built with the melt below and the stay block
(one row per person, no alternative-varying columns) is built directly, then both are concatenated.


In [ ]:
def wide_to_long(df, id_col, alt_list, alt_name, varying, sep, alt_is_prefix):
    assert alt_is_prefix and sep == "_"
    varying_cols = {f"{alt}{sep}{suf}" for alt in alt_list for suf in varying}
    id_vars = [c for c in df.columns if c not in varying_cols]
    frames = []
    for alt in alt_list:
        rename = {f"{alt}{sep}{suf}": suf for suf in varying}
        sub = df[id_vars + list(rename.keys())].rename(columns=rename)
        sub[alt_name] = alt
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)


### Move block: melt + move-only and shared terms

`MOVE_ONLY_TERMS` are structurally zero at `alt=0` (they only make sense as a move decision, e.g.
`destchoice_samestate`). `SHARED_TERMS` hold the move-context value here (e.g. comparing the mover to
the *destination*); the stay block below fills in the stay-context value (comparing to the *origin*)
for the same column names, so concatenating gives each shared coefficient the right value on every row.

`c_proportion_same_race_white` and `c_proportion_same_race_other` from `modeling_mnl.ipynb` are omitted
here (see intro markdown cell) -- their source columns aren't in `INDIV_COLS` and one is populated from a
truncated column-name string in the source notebook.


In [ ]:
move_long = wide_to_long(
    df,
    id_col="person_id",
    alt_list=[f"ALT{i}" for i in range(1, num_alternatives + 1)],
    alt_name="alt_label",
    varying=ALT_VARYING_SUFFIXES,
    sep="_",
    alt_is_prefix=True,
)
move_long["alt"] = move_long["alt_label"].str[len("ALT") :].astype(int)
move_long["choice"] = (move_long["alt"] == move_long["ALT_CHOICE"]).astype(int)

same_state = (move_long["ST"] == move_long["STATE"]).astype(float)
# NAME_NUM.ORIG and ALT{i}_CBSA are factorized against the same CBSA-name codebook
# (see create_estdata.ipynb), so they're directly comparable
same_cbsa = (move_long["NAME_NUM.ORIG"] == move_long["CBSA"]).astype(float)
same_type_t34 = (move_long["TYPE_NUM.ORIG"] == 0).astype(float)
same_type_metro = (move_long["TYPE_NUM.ORIG"] == 1).astype(float)
same_type_nonmetro = (move_long["TYPE_NUM.ORIG"] == 2).astype(float)
alt_type_t34 = (move_long["TYPE"] == 0).astype(float)
alt_type_metro = (move_long["TYPE"] == 1).astype(float)
alt_type_nonmetro = (move_long["TYPE"] == 2).astype(float)

# fixed (coefficient == 1, not estimated) log-population offset -- destination population for move rows.
move_long["log_pop_offset"] = np.log(move_long["TOT_POP"])

# move-only terms
move_long["destchoice_logdist"] = np.log(move_long["DIST"] + 1)
move_long["destchoice_samecbsa"] = same_cbsa
move_long["destchoice_samestate"] = same_state
move_long["destchoice_birthstate"] = (move_long["POBP"] == move_long["STATE"]).astype(
    float
)
# origin area type x destination area type; nonmetro_nonmetro is the omitted reference category,
# matching the commented-out c_destchoice_nonmetro_nonmetro in modeling_mnl.ipynb's cell 22.
move_long["destchoice_T34_T34"] = same_type_t34 * alt_type_t34
move_long["destchoice_T34_metro"] = same_type_t34 * alt_type_metro
move_long["destchoice_T34_nonmetro"] = same_type_t34 * alt_type_nonmetro
move_long["destchoice_metro_T34"] = same_type_metro * alt_type_t34
move_long["destchoice_metro_metro"] = same_type_metro * alt_type_metro
move_long["destchoice_metro_nonmetro"] = same_type_metro * alt_type_nonmetro
move_long["destchoice_nonmetro_T34"] = same_type_nonmetro * alt_type_t34
move_long["destchoice_nonmetro_metro"] = same_type_nonmetro * alt_type_metro

# shared terms (move-context value: comparing the mover to the destination)
move_long["proportion_same_age_18_34"] = (
    move_long["AGE_18_34"] * move_long["OWN_AGE_PCT"]
)
move_long["proportion_same_age_35_64"] = (
    move_long["AGE_35_64"] * move_long["OWN_AGE_PCT"]
)
move_long["proportion_same_age_65_plus"] = (
    move_long["AGE_OVER_65"] * move_long["OWN_AGE_PCT"]
)
move_long["proportion_hh_with_children_if_have_children"] = (
    move_long["HH_WITH_CHILD_PCT"] * move_long["CHILD"]
)
move_long["proportion_college_if_in_college"] = (
    move_long["IN_COLLEGE"] * move_long["COLLEGE_PCT"]
)
move_long["proportion_foreign_if_foreign"] = (
    move_long["FOREIGN"] * move_long["FOREIGN_BORN_PCT"]
)
move_long["median_hh_income_in_tens_of_thousands"] = move_long["HH_MED_INC"] / 10_000
move_long["median_house_value_over_median_income"] = move_long[
    "MED_HOUSE_VALUE_OVER_MED_HH_INC"
]
move_long["median_gross_rent_percentage_hh_inc"] = move_long["MED_RENT_PCT_HH_INC"]
move_long["unemp_rate"] = move_long["UNEMP_RATE"]
move_long["vacancy_rate"] = move_long["HOUSE_VACANCY_PCT"]
move_long["median_travel_time"] = move_long["MED_TRAVEL_TIME"]
move_long["proportion_alt_commute"] = move_long["ALT_COMMUTE_PCT"]
move_long["proportion_ent"] = move_long["ENT_JOBS_PCT"]
move_long["proportion_ent_18_34"] = move_long["AGE_18_34"] * move_long["ENT_JOBS_PCT"]
move_long["proportion_ent_35_64"] = move_long["AGE_35_64"] * move_long["ENT_JOBS_PCT"]
move_long["proportion_also_mil"] = move_long["IN_MILITARY"] * move_long["MIL_PCT"]
# c_proportion_same_naics_goods_trade uses ALT{i}_OWN_NAICS_GROUP_PCT here, not the mangled
# f"{alt}OWN_GROUPOWN_NAICS_GROUP_PCT_PCT" that modeling_mnl.ipynb's cell 23 currently has --
# see the intro markdown cell.
move_long["proportion_same_naics_govt"] = (
    move_long["NAICS_GOVT"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_goods_trade"] = (
    move_long["NAICS_GOODS_TRADE"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_license"] = (
    move_long["NAICS_LICENSE"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_high_ed"] = (
    move_long["NAICS_HIGH_ED"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_agr_ext"] = (
    move_long["NAICS_AGR_EXT"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_race_black"] = (
    move_long["BLACK"] * move_long["OWN_RACE_ETH_PCT"]
)
move_long["proportion_same_race_aapi"] = (
    move_long["AAPI"] * move_long["OWN_RACE_ETH_PCT"]
)
move_long["proportion_same_race_indian"] = (
    move_long["INDIAN"] * move_long["OWN_RACE_ETH_PCT"]
)
move_long["proportion_also_latino"] = (
    move_long["LATINO"] * move_long["OWN_RACE_ETH_PCT"]
)

MOVE_ONLY_TERMS = [
    "destchoice_logdist",
    "destchoice_samecbsa",
    "destchoice_samestate",
    "destchoice_birthstate",
    "destchoice_T34_T34",
    "destchoice_T34_metro",
    "destchoice_T34_nonmetro",
    "destchoice_metro_T34",
    "destchoice_metro_metro",
    "destchoice_metro_nonmetro",
    "destchoice_nonmetro_T34",
    "destchoice_nonmetro_metro",
]
SHARED_TERMS = [
    "proportion_same_age_18_34",
    "proportion_same_age_35_64",
    "proportion_same_age_65_plus",
    "proportion_hh_with_children_if_have_children",
    "proportion_college_if_in_college",
    "proportion_foreign_if_foreign",
    "median_hh_income_in_tens_of_thousands",
    "median_house_value_over_median_income",
    "median_gross_rent_percentage_hh_inc",
    "unemp_rate",
    "vacancy_rate",
    "median_travel_time",
    "proportion_alt_commute",
    "proportion_ent",
    "proportion_ent_18_34",
    "proportion_ent_35_64",
    "proportion_also_mil",
    "proportion_same_naics_govt",
    "proportion_same_naics_goods_trade",
    "proportion_same_naics_license",
    "proportion_same_naics_high_ed",
    "proportion_same_naics_agr_ext",
    "proportion_same_race_black",
    "proportion_same_race_aapi",
    "proportion_same_race_indian",
    "proportion_also_latino",
]
move_long = move_long[
    ["person_id", "alt", "choice", "log_pop_offset"] + MOVE_ONLY_TERMS + SHARED_TERMS
]
print(move_long.shape)


### Stay block: one `alt=0` row per person, direct (no melt -- different variables entirely, not an
alternative-varying reshape)


In [ ]:
stay = df.copy()
stay["stay"] = 1.0  # c_stay: the stay alternative-specific constant
stay["stay_age_18_22"] = stay["AGE_18_22"]
stay["stay_age_23_29"] = stay["AGE_23_29"]
stay["stay_age_30_39"] = stay["AGE_30_39"]
stay["stay_age_40_49"] = stay["AGE_40_49"]
stay["stay_age_50_64"] = stay["AGE_50_64"]

stay["stay_child_under_6"] = stay["CHILD_UNDER_6"]
stay["stay_child_6_to_17"] = stay["CHILD_6_TO_17"]

stay["stay_married_more_than_year"] = stay["MARRIED_MORE_THAN_YEAR"]
stay["stay_married_less_than_year"] = stay["RECENTLY_MARRIED"]
stay["stay_recently_divorced_or_widowed"] = stay["RECENTLY_WIDOWED_OR_DIVORCED"]
stay["stay_2work_mar"] = stay["WORK2_MAR"]
stay["stay_single_parent"] = stay["SINGLE_PARENT"]

stay["stay_edu_college"] = stay["EDU_BACHELORS"]
stay["stay_edu_high"] = stay["EDU_HIGH"]

stay["stay_in_college"] = stay["IN_COLLEGE"]
stay["stay_foreign"] = stay["FOREIGN"]

# NOTE: this assumes that NAICS code stays constant between the origin and destination
stay["stay_mil"] = stay["IN_MILITARY"]
stay["stay_naics_govt"] = stay["NAICS_GOVT"]
stay["stay_naics_goods_trade"] = stay["NAICS_GOODS_TRADE"]
stay["stay_naics_license"] = stay["NAICS_LICENSE"]
stay["stay_naics_high_ed"] = stay["NAICS_HIGH_ED"]
stay["stay_naics_agr_ext"] = stay["NAICS_AGR_EXT"]

STAY_ONLY_TERMS = [
    "stay",
    "stay_age_18_22",
    "stay_age_23_29",
    "stay_age_30_39",
    "stay_age_40_49",
    "stay_age_50_64",
    "stay_child_under_6",
    "stay_child_6_to_17",
    "stay_married_more_than_year",
    "stay_married_less_than_year",
    "stay_recently_divorced_or_widowed",
    "stay_2work_mar",
    "stay_single_parent",
    "stay_edu_college",
    "stay_edu_high",
    "stay_in_college",
    "stay_foreign",
    "stay_mil",
    "stay_naics_govt",
    "stay_naics_goods_trade",
    "stay_naics_license",
    "stay_naics_high_ed",
    "stay_naics_agr_ext",
]

HH_INCOME_COL = (
    "Median Household Income (In 2018 Inflation Adjusted Dollars)."
    "Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG"
)

# fixed (coefficient == 1, not estimated) log-population offset -- origin population for the stay row.
stay["log_pop_offset"] = np.log(
    stay["Total Population.Total Population.SE_A00001_001.ORIG"]
)

# shared terms (stay-context value: comparing the person to their origin)
stay["proportion_same_age_18_34"] = (
    stay["Proportion of people 18-34.ORIG"] * stay["AGE_18_34"]
)
stay["proportion_same_age_35_64"] = (
    stay["Proportion of people 35-64.ORIG"] * stay["AGE_35_64"]
)
stay["proportion_same_age_65_plus"] = (
    stay["Proportion of people 65+.ORIG"] * stay["AGE_OVER_65"]
)
stay["proportion_hh_with_children_if_have_children"] = (
    stay["Proportion of households with children.ORIG"] * stay["CHILD"]
)
stay["proportion_college_if_in_college"] = (
    stay["Proportion of people in college.ORIG"] * stay["IN_COLLEGE"]
)
stay["proportion_foreign_if_foreign"] = (
    stay["Proportion foreign born.ORIG"] * stay["FOREIGN"]
)
stay["median_hh_income_in_tens_of_thousands"] = stay[HH_INCOME_COL] / 10_000
stay["median_house_value_over_median_income"] = stay[
    "Median house value over median household income.ORIG"
]
stay["median_gross_rent_percentage_hh_inc"] = stay[
    "Median gross rent as a percentage of household income.ORIG"
]
stay["unemp_rate"] = stay["Unemployment rate.ORIG"]
stay["vacancy_rate"] = stay["House vacancy proportion.ORIG"]
stay["median_travel_time"] = stay["Median travel time.ORIG"]
stay["proportion_alt_commute"] = stay["Proportion alternative commute.ORIG"]
stay["proportion_ent"] = stay["Proportion of entertainment jobs.ORIG"]
stay["proportion_ent_18_34"] = (
    stay["AGE_18_34"] * stay["Proportion of entertainment jobs.ORIG"]
)
stay["proportion_ent_35_64"] = (
    stay["AGE_35_64"] * stay["Proportion of entertainment jobs.ORIG"]
)
stay["proportion_also_mil"] = (
    stay["Proportion of people in military.ORIG"] * stay["IN_MILITARY"]
)
stay["proportion_same_naics_govt"] = (
    stay["NAICS_GROUP_PCT_GOVT.ORIG"] * stay["NAICS_GOVT"]
)
stay["proportion_same_naics_goods_trade"] = (
    stay["NAICS_GROUP_PCT_GOODS_TRADE.ORIG"] * stay["NAICS_GOODS_TRADE"]
)
stay["proportion_same_naics_license"] = (
    stay["NAICS_GROUP_PCT_LICENSE.ORIG"] * stay["NAICS_LICENSE"]
)
stay["proportion_same_naics_high_ed"] = (
    stay["NAICS_GROUP_PCT_HIGH_ED.ORIG"] * stay["NAICS_HIGH_ED"]
)
stay["proportion_same_naics_agr_ext"] = (
    stay["NAICS_GROUP_PCT_AGR_EXT.ORIG"] * stay["NAICS_AGR_EXT"]
)
stay["proportion_same_race_black"] = (
    stay["Proportion of people Black.ORIG"] * stay["BLACK"]
)
stay["proportion_same_race_aapi"] = (
    stay["Proportion of people AAPI.ORIG"] * stay["AAPI"]
)
stay["proportion_same_race_indian"] = (
    stay["Proportion of people Indian.ORIG"] * stay["INDIAN"]
)
stay["proportion_also_latino"] = (
    stay["Proportion of people Latino.ORIG"] * stay["LATINO"]
)

stay["alt"] = 0
stay["choice"] = (stay["ALT_CHOICE"] == 0).astype(int)
stay = stay[
    ["person_id", "alt", "choice", "log_pop_offset"] + STAY_ONLY_TERMS + SHARED_TERMS
]
print(stay.shape)


### Concatenate and build torch-choice tensors

Each block only has real values for its own terms; the other block's terms are structurally `0` on its
rows (not missing). Both blocks always contribute a value for the shared coefficient names and the
`log_pop_offset`. After sorting by `(person_id, alt)`, each person occupies a contiguous run of
`num_alts` rows in a fixed alt order, so the covariate matrix reshapes directly into torch-choice's
`(num_sessions, num_items, num_params)` layout without needing `EasyDatasetWrapper`'s (much slower,
per-column `pivot`-based) reshape.


In [ ]:
long_df = pd.concat([stay, move_long], ignore_index=True, sort=False)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS
long_df[varnames + ["log_pop_offset"]] = long_df[varnames + ["log_pop_offset"]].fillna(
    0.0
)
long_df = long_df.sort_values(["person_id", "alt"]).reset_index(drop=True)

num_persons = df["person_id"].nunique()
num_alts = num_alternatives + 1  # 51: alt=0 (stay) + alt=1..50 (move)
assert len(long_df) == num_persons * num_alts, (len(long_df), num_persons * num_alts)

X = (
    long_df[varnames]
    .to_numpy(dtype=np.float32)
    .reshape(num_persons, num_alts, len(varnames))
)
offset = (
    long_df["log_pop_offset"].to_numpy(dtype=np.float32).reshape(num_persons, num_alts)
)
choice_2d = long_df["choice"].to_numpy().reshape(num_persons, num_alts)
assert (choice_2d.sum(axis=1) == 1).all(), (
    "each person must choose exactly one alternative"
)
item_index = torch.from_numpy(choice_2d.argmax(axis=1)).long()

itemsession_x = torch.from_numpy(X)
offset_tensor = torch.from_numpy(offset)

dataset = ChoiceDataset(
    item_index=item_index,
    num_items=num_alts,
    num_sessions=num_persons,
    itemsession_x=itemsession_x,
)
dataset


### `OffsetConditionalLogitModel`: adding the un-parameterized `log(population)` term

`ConditionalLogitModel` assigns exactly one estimated coefficient per input column -- there's no way to
add a column to utility with a coefficient pinned at `1` (not trained), which is what Biogeme's bare
`log(Variable(...))` calls in `V[0]`/`V[i]` do (and what xlogit's `addit=` argument does). This subclass
adds `log_pop_offset` to the utility after the model's usual coefficient-weighted terms, as a registered
buffer: it moves with `.to(device)`, and being a buffer rather than a parameter, it's excluded from both
training (`.parameters()`) and the coefficient report.


In [ ]:
class OffsetConditionalLogitModel(ConditionalLogitModel):
    def __init__(self, *args, offset: torch.Tensor, **kwargs):
        super().__init__(*args, **kwargs)
        self.register_buffer("_offset", offset)

    def forward(self, batch, manual_coef_value_dict=None):
        total_utility = super().forward(batch, manual_coef_value_dict)
        return total_utility + self._offset[batch.session_index]


### Fitting

One coefficient per `varnames` entry (`"itemsession_x": "constant"`), no `intercept` key -- `stay` in
`STAY_ONLY_TERMS` is already an explicit ASC for staying, matching `fit_intercept=False` in the earlier
xlogit port / Biogeme not adding an implicit ASC of its own.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = OffsetConditionalLogitModel(
    coef_variation_dict={"itemsession_x": "constant"},
    num_param_dict={"itemsession_x": len(varnames)},
    num_items=num_alts,
    offset=offset_tensor,
).to(device)
dataset = dataset.to(device)


In [ ]:
# NOTE: Adam, not BFGS/LBFGS -- see the optimizer note in the intro markdown cell. Full-batch
# (batch_size=-1) to match Biogeme/xlogit's full-batch MLE convention. Tune num_epochs/learning_rate;
# this is a starting point, not a verified-converged setting.
trained_model = tc_run(
    model,
    dataset,
    batch_size=-1,
    num_epochs=1000,
    learning_rate=0.05,
    model_optimizer="Adam",
    report_frequency=25,
    compute_std=True,
)


### Readable coefficient table

`run()`'s printed report names coefficients generically (`itemsession_x[constant]_0`, `_1`, ...) since
`ConditionalLogitModel` doesn't know about `varnames`. This re-attaches the actual variable names, in the
same order as `varnames`, for readability (matching Biogeme's/xlogit's named per-coefficient output) --
without standard errors, which would mean reimplementing `run()`'s internal Hessian computation with named
outputs; the printed report above already has those, just under the generic `_i` names.


In [ ]:
coef = trained_model.get_coefficient("itemsession_x[constant]").cpu().numpy()
pd.Series(coef, index=varnames, name="Estimation")
